In [1]:
import os
import json
import math
import random
from collections import Counter

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

## Config

In [2]:
SEED = 42

TRAIN_PATH = "../data/train.json"
VAL_PATH = "../data/val.json"
TEST_PATH = "../data/test.json"

VOCAB_SAVE_PATH = "../experiments/saved_models/gru_vocab.json"
MODEL_SAVE_PATH = "../experiments/saved_models/gru_lm.pt"
METRICS_SAVE_PATH = "../results/tables/gru_metrics.json"
# BATCH_SIZE = 32
# EMB_DIM = 128
# HIDDEN_DIM = 256
# NUM_LAYERS = 1
# DROPOUT = 0.1
# LR = 1e-3
# EPOCHS = 5
# MAX_LEN = 30
# MIN_FREQ = 2
BATCH_SIZE = 16
EMB_DIM = 64
HIDDEN_DIM = 128
NUM_LAYERS = 1
DROPOUT = 0.1
LR = 1e-3
EPOCHS = 3
MAX_LEN = 20
MIN_FREQ = 5

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cpu'

## Reproducibility

In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

## Load data

In [4]:
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return pd.DataFrame(rows)

train_df = load_jsonl(TRAIN_PATH)
val_df = load_jsonl(VAL_PATH)
test_df = load_jsonl(TEST_PATH)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

train_df = train_df.sample(n=min(10000, len(train_df)), random_state=SEED).reset_index(drop=True)
val_df = val_df.sample(n=min(2000, len(val_df)), random_state=SEED).reset_index(drop=True)
test_df = test_df.sample(n=min(2000, len(test_df)), random_state=SEED).reset_index(drop=True)

print("Reduced train:", len(train_df))
print("Reduced val:", len(val_df))
print("Reduced test:", len(test_df))

train_df.head()

Train: 47416
Val: 5927
Test: 5928
Reduced train: 10000
Reduced val: 2000
Reduced test: 2000


,sentence,tokens,langs,switches
0,"The racial makeup of the township was 92,93% б...","[The, racial, makeup, of, the, township, was, ...","[EN, EN, EN, EN, EN, EN, EN, EN, RU, EN, RU, R...","[0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, ..."
1,"My friends are not но им нравится то, что я де...","[My, friends, are, not, но, им, нравится, то,,...","[EN, EN, EN, EN, RU, RU, RU, RU, RU, RU, RU]","[0, 0, 0, 1, 0, 0, 0, 0, 0, 0]"
2,"It highlights problems of prospecting, develop...","[It, highlights, problems, of, prospecting,, d...","[EN, EN, EN, EN, EN, EN, EN, EN, EN, EN, EN, E...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,"And what does a vicar что делает викарий, когд...","[And, what, does, a, vicar, что, делает, викар...","[EN, EN, EN, EN, EN, RU, RU, RU, RU, RU, RU]","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0]"
4,An effort should be made to generalize the use...,"[An, effort, should, be, made, to, generalize,...","[EN, EN, EN, EN, EN, EN, EN, EN, EN, EN, RU, R...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ..."


## Check one sample

In [5]:
sample = train_df.iloc[0]
print(sample["sentence"])
print(sample["tokens"])
print(sample["langs"])
print(sample["switches"])

The racial makeup of the township was 92,93% белых, 3,89% коренных американцев, 1,06% азиатов и 2,12% приходится на две или более других рас.
['The', 'racial', 'makeup', 'of', 'the', 'township', 'was', '92,93%', 'белых,', '3,89%', 'коренных', 'американцев,', '1,06%', 'азиатов', 'и', '2,12%', 'приходится', 'на', 'две', 'или', 'более', 'других', 'рас.']
['EN', 'EN', 'EN', 'EN', 'EN', 'EN', 'EN', 'EN', 'RU', 'EN', 'RU', 'RU', 'EN', 'RU', 'RU', 'EN', 'RU', 'RU', 'RU', 'RU', 'RU', 'RU', 'RU']
[0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0]


## Build vocabulary

In [6]:
SPECIAL_TOKENS = ["<pad>", "<unk>", "<bos>", "<eos>"]

def build_vocab(token_lists, min_freq=2):
    counter = Counter()
    for tokens in token_lists:
        counter.update(tokens)

    vocab = SPECIAL_TOKENS.copy()
    for token, freq in counter.items():
        if freq >= min_freq:
            vocab.append(token)

    stoi = {token: idx for idx, token in enumerate(vocab)}
    itos = {idx: token for token, idx in stoi.items()}
    return vocab, stoi, itos, counter

vocab, stoi, itos, counter = build_vocab(train_df["tokens"], min_freq=MIN_FREQ)

print("Vocab size:", len(vocab))
print("Most common:", counter.most_common(20))

Vocab size: 3906
Most common: [('the', 4919), ('of', 2858), ('и', 2573), ('в', 2366), ('to', 2320), ('and', 2000), ('in', 1335), ('a', 1286), ('I', 1281), ('The', 967), ('на', 950), ('не', 940), ('that', 901), ('you', 896), ('is', 844), ('-', 840), ('с', 797), ('for', 772), ('on', 712), ('что', 658)]


## Save vocabulary

In [7]:
os.makedirs("../experiments/saved_models", exist_ok=True)

with open(VOCAB_SAVE_PATH, "w", encoding="utf-8") as f:
    json.dump(stoi, f, ensure_ascii=False, indent=2)

print("Saved vocab to:", VOCAB_SAVE_PATH)

Saved vocab to: ../experiments/saved_models/gru_vocab.json


## Encode tokens

In [8]:
PAD_ID = stoi["<pad>"]
UNK_ID = stoi["<unk>"]
BOS_ID = stoi["<bos>"]
EOS_ID = stoi["<eos>"]

def encode_tokens(tokens, stoi, max_len=30):
    token_ids = [stoi.get(tok, UNK_ID) for tok in tokens]
    
    # truncate so that BOS/EOS still fit
    token_ids = token_ids[: max_len - 2]
    
    input_ids = [BOS_ID] + token_ids
    target_ids = token_ids + [EOS_ID]
    
    return input_ids, target_ids

## Dataset

In [9]:
class CodeSwitchLMDataset(Dataset):
    def __init__(self, df, stoi, max_len=30):
        self.samples = []
        
        for _, row in df.iterrows():
            tokens = row["tokens"]
            switches = row["switches"]
            
            input_ids, target_ids = encode_tokens(tokens, stoi, max_len=max_len)
            
            # truncate switches to fit truncated token sequence
            real_token_count = len(target_ids) - 1  # excludes eos
            switches = switches[: max(0, real_token_count - 1)]
            
            self.samples.append({
                "input_ids": input_ids,
                "target_ids": target_ids,
                "tokens": tokens[:max_len-2],
                "switches": switches
            })
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        return self.samples[idx]

train_dataset = CodeSwitchLMDataset(train_df, stoi, max_len=MAX_LEN)
val_dataset = CodeSwitchLMDataset(val_df, stoi, max_len=MAX_LEN)
test_dataset = CodeSwitchLMDataset(test_df, stoi, max_len=MAX_LEN)

len(train_dataset), len(val_dataset), len(test_dataset)

(10000, 2000, 2000)

## Collate function

In [10]:
def collate_fn(batch):
    max_input_len = max(len(item["input_ids"]) for item in batch)
    max_target_len = max(len(item["target_ids"]) for item in batch)

    batch_input_ids = []
    batch_target_ids = []
    batch_lengths = []
    batch_switches = []

    for item in batch:
        input_ids = item["input_ids"]
        target_ids = item["target_ids"]

        input_pad = input_ids + [PAD_ID] * (max_input_len - len(input_ids))
        target_pad = target_ids + [PAD_ID] * (max_target_len - len(target_ids))

        batch_input_ids.append(input_pad)
        batch_target_ids.append(target_pad)
        batch_lengths.append(len(input_ids))
        batch_switches.append(item["switches"])

    return {
        "input_ids": torch.tensor(batch_input_ids, dtype=torch.long),
        "target_ids": torch.tensor(batch_target_ids, dtype=torch.long),
        "lengths": torch.tensor(batch_lengths, dtype=torch.long),
        "switches": batch_switches
    }

## DataLoaders

In [11]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0
)

## Define GRU Language Model

In [12]:
class GRULanguageModel(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, num_layers=1, dropout=0.1):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_ID)
        self.gru = nn.GRU(
            input_size=emb_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_ids):
        x = self.embedding(input_ids)         # [B, T, E]
        outputs, _ = self.gru(x)              # [B, T, H]
        outputs = self.dropout(outputs)
        logits = self.fc(outputs)             # [B, T, V]
        return logits

## Initialize model

In [13]:
model = GRULanguageModel(
    vocab_size=len(vocab),
    emb_dim=EMB_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT
).to(DEVICE)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

model

GRULanguageModel(
  (embedding): Embedding(3906, 64, padding_idx=0)
  (gru): GRU(64, 128, batch_first=True)
  (dropout): Dropout(p=0.1, inplace=False)
  (fc): Linear(in_features=128, out_features=3906, bias=True)
)

## Perplexity helper

In [14]:
def perplexity_from_loss(loss):
    return math.exp(loss)

## Switch-point helper

In [15]:
def token_lang_from_id(token_id, itos):
    token = itos.get(token_id, "<unk>")
    for ch in token:
        if "а" <= ch <= "я" or "А" <= ch <= "Я":
            return "RU"
    return "EN"

## Evaluation function

In [16]:
def evaluate_model(model, data_loader, criterion, device, itos):
    model.eval()
    
    total_loss = 0.0
    total_batches = 0
    
    switch_correct = 0
    switch_total = 0
    
    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Evaluating", leave=False):
            input_ids = batch["input_ids"].to(device)
            target_ids = batch["target_ids"].to(device)

            logits = model(input_ids)
            vocab_size = logits.size(-1)

            loss = criterion(
                logits.view(-1, vocab_size),
                target_ids.view(-1)
            )
            
            total_loss += loss.item()
            total_batches += 1

            preds = logits.argmax(dim=-1).cpu().numpy()
            gold_input = batch["input_ids"].cpu().numpy()
            gold_target = batch["target_ids"].cpu().numpy()

            # switch evaluation only on real token transitions
            for i in range(len(batch["switches"])):
                gold_switches = batch["switches"][i]
                
                for j, gold_switch in enumerate(gold_switches):
                    current_token_id = gold_input[i][j + 1]   # tok_j+1
                    pred_next_token_id = preds[i][j + 1]      # predicted tok_j+2
                    
                    current_lang = token_lang_from_id(int(current_token_id), itos)
                    pred_next_lang = token_lang_from_id(int(pred_next_token_id), itos)

                    pred_switch = 1 if current_lang != pred_next_lang else 0
                    
                    switch_correct += int(pred_switch == gold_switch)
                    switch_total += 1

    avg_loss = total_loss / max(1, total_batches)
    ppl = perplexity_from_loss(avg_loss)
    switch_acc = switch_correct / max(1, switch_total)
    

    return {
        "loss": avg_loss,
        "perplexity": ppl,
        "switch_accuracy": switch_acc
    }

## Training function

In [17]:
def train_one_epoch(model, data_loader, criterion, optimizer, device):
    model.train()
    
    total_loss = 0.0
    total_batches = 0

    for batch in tqdm(data_loader, desc="Training", leave=False):
        input_ids = batch["input_ids"].to(device)
        target_ids = batch["target_ids"].to(device)

        optimizer.zero_grad()

        logits = model(input_ids)
        vocab_size = logits.size(-1)

        loss = criterion(
            logits.view(-1, vocab_size),
            target_ids.view(-1)
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        total_batches += 1

    avg_loss = total_loss / max(1, total_batches)
    return avg_loss

## Train loop

In [18]:
best_val_loss = float("inf")
history = []

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_metrics = evaluate_model(model, val_loader, criterion, DEVICE, itos)

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_perplexity": perplexity_from_loss(train_loss),
        "val_loss": val_metrics["loss"],
        "val_perplexity": val_metrics["perplexity"],
        "val_switch_accuracy": val_metrics["switch_accuracy"]
    }
    history.append(row)

    print(f"Epoch {epoch}")
    print(f"  train_loss: {train_loss:.4f}")
    print(f"  train_ppl:  {perplexity_from_loss(train_loss):.4f}")
    print(f"  val_loss:   {val_metrics['loss']:.4f}")
    print(f"  val_ppl:    {val_metrics['perplexity']:.4f}")
    print(f"  val_switch_acc: {val_metrics['switch_accuracy']:.4f}")

    if val_metrics["loss"] < best_val_loss:
        best_val_loss = val_metrics["loss"]
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print("  Saved best model!")

Epoch 1
  train_loss: 4.8782
  train_ppl:  131.3979
  val_loss:   4.3268
  val_ppl:    75.6989
  val_switch_acc: 0.7536
  Saved best model!


Epoch 2
  train_loss: 4.4075
  train_ppl:  82.0675
  val_loss:   4.1326
  val_ppl:    62.3404
  val_switch_acc: 0.7536
  Saved best model!


Epoch 3
  train_loss: 4.1948
  train_ppl:  66.3426
  val_loss:   4.0191
  val_ppl:    55.6516
  val_switch_acc: 0.7538
  Saved best model!


## Training history

In [19]:
history_df = pd.DataFrame(history)
history_df

,epoch,train_loss,train_perplexity,val_loss,val_perplexity,val_switch_accuracy
0,1,4.878230,131.397898,4.326763,75.698864,0.753625
1,2,4.407543,82.067548,4.132609,62.340363,0.753625
2,3,4.194833,66.342635,4.019111,55.651593,0.753769


## Final test evaluation

In [20]:
model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=DEVICE))

test_metrics = evaluate_model(model, test_loader, criterion, DEVICE, itos)
test_metrics

{'loss': 4.030971662521362,
 'perplexity': 56.31560443336087,
 'switch_accuracy': 0.7478090575275398}

## Save metrics

In [21]:
os.makedirs("../results/tables", exist_ok=True)

final_metrics = {
    "model": "GRU_LM",
    "vocab_size": len(vocab),
    "embedding_dim": EMB_DIM,
    "hidden_dim": HIDDEN_DIM,
    "num_layers": NUM_LAYERS,
    "dropout": DROPOUT,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "epochs": EPOCHS,
    "best_val_loss": best_val_loss,
    "test_loss": test_metrics["loss"],
    "test_perplexity": test_metrics["perplexity"],
    "test_switch_accuracy": test_metrics["switch_accuracy"]
}

with open(METRICS_SAVE_PATH, "w", encoding="utf-8") as f:
    json.dump(final_metrics, f, ensure_ascii=False, indent=2)

print("Saved metrics to:", METRICS_SAVE_PATH)
print(json.dumps(final_metrics, ensure_ascii=False, indent=2))

Saved metrics to: ../results/tables/gru_metrics.json
{
  "model": "GRU_LM",
  "vocab_size": 3906,
  "embedding_dim": 64,
  "hidden_dim": 128,
  "num_layers": 1,
  "dropout": 0.1,
  "batch_size": 16,
  "lr": 0.001,
  "epochs": 3,
  "best_val_loss": 4.019110696792603,
  "test_loss": 4.030971662521362,
  "test_perplexity": 56.31560443336087,
  "test_switch_accuracy": 0.7478090575275398
}


## Save training history CSV

In [22]:
history_df.to_csv("../results/tables/gru_training_history.csv", index=False)
print("Saved training history.")

Saved training history.


The compact GRU language model was trained for 3 epochs on a reduced dataset due to local computational limits. The validation perplexity decreased from 75.70 to 55.65, showing that the model learned useful sequential patterns. On the test set, the GRU achieved perplexity 56.32 and switch-point accuracy 0.748. This result will be used as the recurrent baseline for comparison with the CNN language model.